
# Assignment: Shadow-Invariant Vision with OpenCV and Kornia

## Motivation

A shadow can create a strong image boundary even though there is no corresponding boundary in scene geometry or material.

Before beginning, review the following example:

<blockquote class="twitter-tweet" data-media-max-width="560">
<p lang="en" dir="ltr">
Fsd 14.2.2.5 swerves for a shadow. I disengaged and reported the issue
<a href="https://t.co/7bse3vKymK">pic.twitter.com/7bse3vKymK</a>
</p>
&mdash; Elias Martinez (@EliasMartinez)
<a href="https://x.com/EliasMartinez/status/2034946481744929113?ref_src=twsrc%5Etfw">March 20, 2026</a>
</blockquote>

<script async src="https://platform.x.com/widgets.js" charset="utf-8"></script>

The central question of this assignment is:

> Can we construct an image representation that responds strongly to changes in surface reflectance but weakly to changes caused only by illumination?

You will implement a shadow-invariant representation inspired by Chapter 10 of Peter Corke's *Robotics, Vision & Control*.

[Download Chapter 10, Light and Color (PDF, 40 pages)](https://artifacts.aegeanai.com/pdf/corke-rvc3/ch10-light-and-color.pdf)

Reference notebook:

https://github.com/petercorke/RVC3-python/blob/main/notebooks/chap10.ipynb

Focus on:

- §10.1 Spectral Representation of Light
- §10.2 Color
- §10.4 Application: Color Images
- §10.4.2 Shadow Removal

You will first implement the relevant computations with NumPy and OpenCV, then port the final shadow-invariant pipeline to PyTorch/Kornia.



## Learning Objectives

By the end of this assignment, you should be able to:

1. Explain why shadows alter image measurements without altering surface material.
2. Relate illumination spectra to measured color.
3. Compare RGB, HSV, and Lab for illuminated and shadowed regions.
4. Construct a log-chromaticity representation.
5. Compute a one-dimensional shadow-invariant image.
6. Quantitatively evaluate shadow suppression.
7. Port the final computation to a PyTorch/Kornia tensor pipeline.



# 1. Development Environment

Use the course development environment:

https://github.com/pantelis/eng-ai-agents

Open the repository in VS Code as a Dev Container.

Inside the container run:

```bash
make start
make install-notebooks
source .venv/bin/activate
```

`make start` alone installs only the base dependencies. `make install-notebooks` adds OpenCV, PyTorch and Kornia, which the sections below use.

GPU acceleration is not required for full credit.


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import kornia
import time

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Kornia:", kornia.__version__)
print("CUDA available:", torch.cuda.is_available())



# 2. Why Does a Shadow Change an Image?

A simplified image-formation model is

$$
I_k =
\int E(\lambda)R(\lambda)Q_k(\lambda)\,d\lambda,
$$

where

- $E(\lambda)$ is illumination,
- $R(\lambda)$ is surface reflectance,
- $Q_k(\lambda)$ is the spectral response of camera channel $k$.

The key observation is:

$$
\text{measured pixel value} \neq \text{surface reflectance alone}.
$$

Measured RGB depends on both the material and the illumination.

In a cast shadow, the material reflectance is approximately unchanged, but the illumination reaching the surface changes.



## 2.1 Blackbody Illumination

Implement Planck's law:

$$
B(\lambda,T)
=
\frac{2hc^2}{\lambda^5}
\frac{1}
{\exp\left(\frac{hc}{\lambda k_BT}\right)-1}.
$$

Plot normalized spectra for:

$$
T_1 = 2600\text{ K},
\qquad
T_2 = 5778\text{ K}.
$$

Answer briefly:

1. Why can the same material produce different RGB values under these two illuminants?
2. In a cast shadow, what changes: reflectance or illumination?


In [ ]:

def blackbody(wavelength_m, temperature_k):
    # TODO: implement Planck's blackbody radiation equation.
    pass

wavelength_nm = np.linspace(380, 780, 400)
wavelength_m = wavelength_nm * 1e-9

# TODO:
# lamp = blackbody(wavelength_m, 2600)
# sun = blackbody(wavelength_m, 5778)
# Normalize each curve and plot both.



### Short Answer

Write your response here.

- Same material, different illuminants:
- What changes in a cast shadow:



# 3. Observe a Shadow in Different Color Spaces

Use Corke's `parks.png` image or another supplied outdoor image containing a strong cast shadow.

Load the image with OpenCV and select two small regions of the same physical surface:

- one directly illuminated,
- one inside the shadow.

You will compare the regions in RGB, HSV, and Lab.


In [ ]:

import os
import urllib.request

# parks.png is the outdoor scene Corke uses in chapter 10. It ships with the
# Machine Vision Toolbox data package (Peter Corke, MIT license).
IMAGE_URL = (
    "https://raw.githubusercontent.com/petercorke/machinevision-toolbox-python/"
    "main/packages/mvtb-data/mvtbdata/images/parks.png"
)
IMAGE_PATH = "parks.png"  # TODO: point this at your own image if you prefer

if not os.path.exists(IMAGE_PATH):
    urllib.request.urlretrieve(IMAGE_URL, IMAGE_PATH)

image_bgr = cv2.imread(IMAGE_PATH)
if image_bgr is None:
    raise FileNotFoundError(
        f"Could not load {IMAGE_PATH}. Check the path, or download the image "
        f"yourself from {IMAGE_URL}."
    )

print("Image shape (H, W, C):", image_bgr.shape)

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(image_rgb)
plt.axis("off")
plt.title("Original image")
plt.show()



## 3.1 Define Illuminated and Shadowed Regions

Choose two rectangular regions containing the same material.

Store them as:

```python
(x1, y1, x2, y2)
```

where `(x1, y1)` is the upper-left corner and `(x2, y2)` is the lower-right corner.


In [ ]:

# TODO: replace with coordinates appropriate for your image.
lit_roi = (0, 0, 50, 50)
shadow_roi = (60, 0, 110, 50)

def crop_roi(image, roi):
    x1, y1, x2, y2 = roi
    return image[y1:y2, x1:x2]

def mean_color(image, roi):
    patch = crop_roi(image, roi)
    return patch.reshape(-1, patch.shape[-1]).mean(axis=0)

# TODO: visualize the chosen rectangles on the image.


In [ ]:

hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)

rgb_lit = mean_color(image_rgb, lit_roi)
rgb_shadow = mean_color(image_rgb, shadow_roi)

hsv_lit = mean_color(hsv, lit_roi)
hsv_shadow = mean_color(hsv, shadow_roi)

lab_lit = mean_color(lab, lit_roi)
lab_shadow = mean_color(lab, shadow_roi)

print("RGB lit:    ", rgb_lit)
print("RGB shadow: ", rgb_shadow)
print("HSV lit:    ", hsv_lit)
print("HSV shadow: ", hsv_shadow)
print("Lab lit:    ", lab_lit)
print("Lab shadow: ", lab_shadow)



### Questions

1. Which components change most strongly across the shadow?
2. Does converting to HSV or Lab completely remove the shadow?
3. Why is separating intensity from chromatic information useful?



# 4. Implement the Shadow-Invariant Image

This is the main part of the assignment.

Do not use Corke's `shadow_invariant()` function.

You will implement:

1. sRGB linearization
2. log chromaticity
3. 2-D projection
4. a 1-D invariant image



## 4.1 Convert sRGB to Linear RGB

Convert the image to floating-point RGB in $[0,1]$.

Implement the inverse sRGB transfer function:

$$
C_{\text{linear}}
=
\begin{cases}
C_{\text{sRGB}}/12.92,
&
C_{\text{sRGB}}\leq0.04045
\\[4pt]
\left(
\frac{C_{\text{sRGB}}+0.055}{1.055}
\right)^{2.4},
&
\text{otherwise}.
\end{cases}
$$


In [ ]:

def srgb_to_linear(rgb):
    # rgb must be float32/float64 in [0, 1]
    # TODO: implement the inverse sRGB transfer function.
    pass

rgb = image_rgb.astype(np.float32) / 255.0
linear_rgb = srgb_to_linear(rgb)

# TODO: display linear_rgb after clipping to [0,1].



## 4.2 Log Chromaticity

For each pixel compute the geometric mean

$$
M=(RGB)^{1/3}.
$$

Then define

$$
\rho_R=\log\frac{R}{M},
\qquad
\rho_G=\log\frac{G}{M},
\qquad
\rho_B=\log\frac{B}{M}.
$$

Use a small $\epsilon$ for numerical stability.

Verify numerically that

$$
\rho_R+\rho_G+\rho_B\approx0.
$$


In [ ]:

def log_chromaticity(linear_rgb, eps=1e-6):
    # TODO:
    # 1. clamp values with eps
    # 2. compute geometric mean M
    # 3. compute rho_R, rho_G, rho_B
    # 4. stack into an H x W x 3 array
    pass

rho = log_chromaticity(linear_rgb)

# TODO: verify rho[...,0] + rho[...,1] + rho[...,2] is approximately zero.



## 4.3 Project to a Two-Dimensional Chromaticity Plane

Use the orthonormal basis

$$
u_1=
\frac{1}{\sqrt{2}}
\begin{bmatrix}
1\\
-1\\
0
\end{bmatrix},
\qquad
u_2=
\frac{1}{\sqrt{6}}
\begin{bmatrix}
1\\
1\\
-2
\end{bmatrix}.
$$

Compute

$$
\chi_1=u_1^T\rho,
\qquad
\chi_2=u_2^T\rho.
$$


In [ ]:

u1 = np.array([1.0, -1.0, 0.0]) / np.sqrt(2.0)
u2 = np.array([1.0, 1.0, -2.0]) / np.sqrt(6.0)

# TODO: compute chi1 and chi2 for all pixels.
chi1 = None
chi2 = None



## 4.4 Plot Illuminated and Shadowed Pixels in Log-Chromaticity Space

Plot pixels from the two selected ROIs in $(\chi_1,\chi_2)$ coordinates.

Use different markers or colors for the illuminated and shadowed regions.

Do the two groups appear displaced primarily along a common direction?


In [ ]:

def roi_values(image_2d, roi):
    x1, y1, x2, y2 = roi
    return image_2d[y1:y2, x1:x2].ravel()

# TODO:
# lit_x = roi_values(chi1, lit_roi)
# lit_y = roi_values(chi2, lit_roi)
# shadow_x = roi_values(chi1, shadow_roi)
# shadow_y = roi_values(chi2, shadow_roi)
#
# plt.scatter(...)



## 4.5 Shadow-Invariant Projection

Reproduce Corke's fixed-angle example using

$$
\theta=0.7\text{ rad}.
$$

Compute

$$
I_{\text{inv}}
=
\chi_1\cos\theta
+
\chi_2\sin\theta.
$$

Normalize only for visualization.


In [ ]:

theta = 0.7

# TODO:
# invariant = chi1 * np.cos(theta) + chi2 * np.sin(theta)

# Normalize for display only:
# invariant_display = cv2.normalize(
#     invariant, None, 0, 255, cv2.NORM_MINMAX
# ).astype(np.uint8)



Display side-by-side:

1. original RGB image,
2. ordinary grayscale image,
3. shadow-invariant image.

Then answer:

1. Which shadow boundaries became weaker?
2. Which physical or material boundaries remained?


In [ ]:

gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

# TODO: create a 1x3 figure showing:
# original RGB
# grayscale
# invariant_display



# 5. Quantitative Evaluation

Use the same illuminated and shadowed ROIs.

For grayscale compute

$$
\Delta_{\text{gray}}
=
|\mu_{\text{lit}}-\mu_{\text{shadow}}|.
$$

For the invariant image compute

$$
\Delta_{\text{inv}}
=
|\mu_{\text{lit}}^{\text{inv}}
-
\mu_{\text{shadow}}^{\text{inv}}|.
$$

A successful representation should approximately satisfy

$$
\Delta_{\text{inv}} < \Delta_{\text{gray}}.
$$


In [ ]:

def mean_scalar(image_2d, roi):
    return roi_values(image_2d, roi).mean()

# TODO:
# delta_gray = ...
# delta_inv = ...
# print(...)



## 5.1 Compare Edges

Use OpenCV Canny on:

1. the ordinary grayscale image,
2. the normalized shadow-invariant image.

Look for one shadow edge and one real material/object edge.

The desired behavior is:

- weaker response at the shadow boundary,
- preservation of real scene boundaries.


In [ ]:

# TODO: tune thresholds if needed.
edges_gray = cv2.Canny(gray, 50, 150)

# TODO:
# edges_inv = cv2.Canny(invariant_display, 50, 150)

# TODO: display both edge maps.



# 6. Port the Final Pipeline to PyTorch and Kornia

The goal of this section is to express the same classical computer-vision pipeline as tensor operations.

This allows:

- batching,
- GPU execution,
- automatic differentiation.

Kornia provides differentiable computer-vision operators that integrate directly with PyTorch computation graphs.


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

rgb_t = (
    torch.from_numpy(image_rgb)
    .permute(2, 0, 1)
    .unsqueeze(0)
    .float()
    .div(255.0)
    .to(device)
)

print("Tensor shape:", rgb_t.shape)
print("Device:", rgb_t.device)



## 6.1 Linear RGB with Kornia

Use:

```python
kornia.color.rgb_to_linear_rgb(...)
```

Then reproduce the same log-chromaticity and projection equations using PyTorch tensor operations.


In [ ]:

linear_t = kornia.color.rgb_to_linear_rgb(rgb_t)

def shadow_invariant_torch(linear_rgb_t, theta=0.7, eps=1e-6):
    # Expected input shape: B x 3 x H x W
    #
    # TODO:
    # 1. clamp channels with eps
    # 2. compute geometric mean
    # 3. compute log chromaticities
    # 4. project using u1 and u2
    # 5. compute invariant = chi1*cos(theta) + chi2*sin(theta)
    #
    # Return shape: B x H x W
    pass

invariant_t = shadow_invariant_torch(linear_t, theta=0.7)



## 6.2 Compare NumPy/OpenCV and PyTorch/Kornia Results

Compute

$$
\mathrm{MAE}
=
\frac{1}{N}
\sum_i
|I_i^{\mathrm{NumPy}}
-
I_i^{\mathrm{PyTorch}}|.
$$

The two implementations should be numerically very similar.


In [ ]:

# TODO:
# invariant_t_np = invariant_t.squeeze(0).detach().cpu().numpy()
# mae = np.mean(np.abs(invariant - invariant_t_np))
# print("MAE:", mae)



## 6.3 Optional GPU Batch Experiment

If CUDA is available, process a batch of 32 identical images.

Report:

- execution device,
- batch shape,
- elapsed time.

No GPU speedup is required for full credit.


In [ ]:

if torch.cuda.is_available():
    batch = rgb_t.repeat(32, 1, 1, 1)
    batch_linear = kornia.color.rgb_to_linear_rgb(batch)

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    # TODO:
    # batch_inv = shadow_invariant_torch(batch_linear, theta=0.7)

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    print("Batch shape:", batch.shape)
    print("Elapsed:", elapsed)
else:
    print("CUDA not available; GPU timing is optional.")



# 7. Discussion

Return to the autonomous-driving example at the beginning.

Answer in approximately one paragraph:

1. Why does the shadow generate strong image structure even though there is no new physical object?
2. What quantity does the invariant projection attempt to suppress?
3. Why can preserving reflectance-related edges while suppressing illumination edges help perception?
4. Why would this classical method still be insufficient by itself for autonomous driving?

Consider geometry, depth, semantics, temporal reasoning, learned representations, and sensor fusion.



# Deliverables

Your notebook must execute from beginning to end, and you must submit it with all outputs saved.

Required outputs:

- two blackbody curves,
- RGB/HSV/Lab comparison,
- log-chromaticity scatter plot,
- shadow-invariant image,
- quantitative shadow comparison,
- OpenCV edge comparison,
- PyTorch/Kornia implementation,
- short final discussion.

## Submitting

Follow the [assignment submission guide](https://aegean.ai/aiml-common/resources/environment/assignment-submission). Commit this notebook with all cell outputs saved, and submit the complete GitHub URL of its directory.

The per-component points are on the assignment page.



# References

1. Peter Corke, *Robotics, Vision & Control: Fundamental Algorithms in Python*, 3rd ed., Springer, Chapter 10. [Download the chapter (PDF)](https://artifacts.aegeanai.com/pdf/corke-rvc3/ch10-light-and-color.pdf)
2. G. D. Finlayson, S. D. Hordley, and M. S. Drew, "Removing Shadows from Images," ECCV, 2002.
3. G. D. Finlayson, M. S. Drew, and C. Lu, "Intrinsic Images by Entropy Minimization," ECCV, 2004.
4. G. D. Finlayson, S. D. Hordley, C. Lu, and M. S. Drew, "On the Removal of Shadows from Images," IEEE TPAMI, 2006.
5. Kornia documentation: https://kornia.readthedocs.io/
6. OpenCV documentation: https://docs.opencv.org/
